# Import e caricamento dataset

In [30]:
from datasets import load_dataset
import pandas as pd
import pickle
import os
from tqdm import tqdm
from collections import Counter
import random
from datetime import datetime, timedelta

Caricamento dataset

In [5]:
dataset_fever = load_dataset("fever", "v1.0", trust_remote_code=True)
print(dataset_fever)

DatasetDict({
    train: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 311431
    })
    labelled_dev: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 37566
    })
    unlabelled_dev: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 19998
    })
    unlabelled_test: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 19998
    })
    paper_dev: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 18999
    })
    paper_test: Dataset({
        features: [

# Studio dataset

Split dataset disponibili

In [6]:
print("Split disponibili: \n", list(dataset_fever.keys()))
print()
for nome_split, dati in dataset_fever.items():
    print(f"  {nome_split}: {len(dati):,} esempi")

Split disponibili: 
 ['train', 'labelled_dev', 'unlabelled_dev', 'unlabelled_test', 'paper_dev', 'paper_test']

  train: 311,431 esempi
  labelled_dev: 37,566 esempi
  unlabelled_dev: 19,998 esempi
  unlabelled_test: 19,998 esempi
  paper_dev: 18,999 esempi
  paper_test: 18,567 esempi


Struttura del dataset

In [7]:
print(dataset_fever["train"][0])

{'id': 75397, 'label': 'SUPPORTS', 'claim': 'Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.', 'evidence_annotation_id': 92206, 'evidence_id': 104971, 'evidence_wiki_url': 'Nikolaj_Coster-Waldau', 'evidence_sentence_id': 7}


In [8]:
print(dataset_fever["train"].features)

{'id': Value(dtype='int32', id=None), 'label': Value(dtype='string', id=None), 'claim': Value(dtype='string', id=None), 'evidence_annotation_id': Value(dtype='int32', id=None), 'evidence_id': Value(dtype='int32', id=None), 'evidence_wiki_url': Value(dtype='string', id=None), 'evidence_sentence_id': Value(dtype='int32', id=None)}


Distribuzione delle label

In [9]:
df = pd.DataFrame(dataset_fever["train"])
df_train = pd.DataFrame(dataset_fever["train"])
df_val = pd.DataFrame(dataset_fever["labelled_dev"]) if "labelled_dev" in dataset_fever else pd.DataFrame(dataset_fever.get("validation", dataset_fever[list(dataset_fever.keys())[1]]))

print(df["label"].value_counts())

label
SUPPORTS           193756
REFUTES             70066
NOT ENOUGH INFO     47609
Name: count, dtype: int64


Lunghezza claim

In [10]:
df["lunghezza_claim"] = df["claim"].apply(lambda x: len(x.split()))
print(df["lunghezza_claim"].describe())

count    311431.000000
mean          7.588089
std           3.049780
min           2.000000
25%           5.000000
50%           7.000000
75%           9.000000
max          65.000000
Name: lunghezza_claim, dtype: float64


Valori mancanti e anomalie

In [11]:
print("Valori mancanti nel training set \n")
print(df_train.isnull().sum().to_string())

print("\nValori mancanti nel validation set \n")
print(df_val.isnull().sum().to_string())

Valori mancanti nel training set 

id                        0
label                     0
claim                     0
evidence_annotation_id    0
evidence_id               0
evidence_wiki_url         0
evidence_sentence_id      0

Valori mancanti nel validation set 

id                        0
label                     0
claim                     0
evidence_annotation_id    0
evidence_id               0
evidence_wiki_url         0
evidence_sentence_id      0


In [12]:
numero_duplicati = df_train["claim"].duplicated().sum()
print(f"Claim duplicate nel training set: {numero_duplicati:,}")

Claim duplicate nel training set: 175,886


# Modifiche dataset

Raggruppamento dataset per claim

In [17]:
df_per_claim = df_train.groupby("id").agg(
    testo_claim = ("claim", "first"),
    label = ("label", "first"),
    numero_evidenze = ("evidence_id", "count"),
    url_documenti = ("evidence_wiki_url", lambda x: list(x.dropna().unique())),
    sentence_ids = ("evidence_sentence_id", list)
).reset_index()

print(f"Righe totali nel dataset: {len(df_train):,}")
print(f"Claim uniche dopo raggruppamento: {len(df_per_claim):,}")

Righe totali nel dataset: 311,431
Claim uniche dopo raggruppamento: 145,449


In [18]:
print("Distribuzione numero evidenze per claim:")
print(df_per_claim["numero_evidenze"].describe().round(2).to_string())
print()
print("Conteggio per numero esatto di evidenze:")
print(df_per_claim["numero_evidenze"].value_counts().sort_index().to_string())

Distribuzione numero evidenze per claim:
count    145449.00
mean          2.14
std           3.66
min           1.00
25%           1.00
50%           1.00
75%           2.00
max         338.00

Conteggio per numero esatto di evidenze:
numero_evidenze
1      98547
2      22067
3       6355
4       4676
5       4115
6       2505
7       1437
8       1036
9        746
10       671
11       460
12       417
13       311
14       285
15       199
16       216
17       161
18       143
19       107
20       115
21        97
22        70
23        84
24        63
25        50
26        41
27        41
28        32
29        32
30        31
31        25
32        28
33        20
34        18
35        16
36        14
37        24
38        13
39        14
40         8
41         9
42        10
43         9
44         6
45         4
46        11
47         6
48         8
49         3
50         5
51         3
52         5
53         6
54         6
55         1
56         3
57         3
58      

In [19]:
print("Distribuzione label su claim uniche:")
conteggio_label = df_per_claim["label"].value_counts()
for label, n in conteggio_label.items():
    p = n / len(df_per_claim) * 100
    print(f"  {label:<20} {n:>7,}  ({p:4.1f}%)")

Distribuzione label su claim uniche:
  SUPPORTS              80,035  (55.0%)
  NOT ENOUGH INFO       35,639  (24.5%)
  REFUTES               29,775  (20.5%)


Analisi corpus wikipedia

In [20]:
pagine_uniche = df_train["evidence_wiki_url"].dropna().unique()
print(f"Pagine Wikipedia uniche citate nel train: {len(pagine_uniche):,}")

df_dev = pd.DataFrame(dataset_fever["labelled_dev"])
pagine_dev = df_dev["evidence_wiki_url"].dropna().unique()
print(f"Pagine Wikipedia uniche citate nel dev: {len(pagine_dev):,}")

overlap = set(pagine_uniche) & set(pagine_dev)
print(f"Pagine in comune tra train e dev: {len(overlap):,}")

Pagine Wikipedia uniche citate nel train: 12,550
Pagine Wikipedia uniche citate nel dev: 2,893
Pagine in comune tra train e dev: 909


# Estrazione corpus

Caricamento corpus wikipedia

In [21]:
corpus_wikipedia = load_dataset("fever", "wiki_pages", split="wikipedia_pages",trust_remote_code=True)
print(f"Pagine totali nel corpus: {len(corpus_wikipedia):,}")

Pagine totali nel corpus: 5,416,537


Visualizzazione primo corpus per analisi struttura

In [22]:
for doc in corpus_wikipedia:
    if doc['id'] and doc['lines']:
        print(f"ID:\n {repr(doc['id'])}")
        print(f"TESTO:\n {repr(doc['text'][:200])}")
        print(f"CONTENUTO GREZZO DI LINES:\n {repr(doc['lines'][:300])}")
        break

ID:
 '1928_in_association_football'
TESTO:
 'The following are the football -LRB- soccer -RRB- events of the year 1928 throughout the world . '
CONTENUTO GREZZO DI LINES:
 '0\tThe following are the football -LRB- soccer -RRB- events of the year 1928 throughout the world .\n1\t'


Funzione per estrarre la frase N da un documento

In [23]:
def estrai_frase(lines: str, sentence_id: int) -> str:
    if not lines:
        return ""
    for riga in lines.split('\n'):
        riga = riga.strip()
        if not riga:
            continue
        parti = riga.split('\t', 1)
        if len(parti) == 2:
            try:
                if int(parti[0]) == sentence_id:
                    return parti[1].strip()
            except ValueError:
                continue
    return ""

In [24]:
for doc in corpus_wikipedia:
    if doc['id'] and doc['lines'] and len(doc['lines']) > 50:
        print(f"Documento: {doc['id']}")
        for i in range(4):
            frase = estrai_frase(doc['lines'], i)
            if frase:
                print(f"  Frase {i}: {frase}")
        break

Documento: 1928_in_association_football
  Frase 0: The following are the football -LRB- soccer -RRB- events of the year 1928 throughout the world .


## Aggiunta timestamp e backlinks

In [25]:
pagine_necessarie = set(df_train['evidence_wiki_url'].dropna().unique())
pagine_necessarie |= set(df_dev['evidence_wiki_url'].dropna().unique())
print(f"Pagine necessarie (train + dev): {len(pagine_necessarie):,}")

Pagine necessarie (train + dev): 14,534


In [34]:
PERCORSO_CORPUS_BASE = "corpus_fever_base.pkl"
PERCORSO_METADATI = "metadati_corpus.pkl"
PERCORSO_CORPUS_ARRICCHITO = "corpus_fever_arricchito.pkl"

In [27]:
if os.path.exists(PERCORSO_CORPUS_BASE):
    print("Corpus base già salvato: caricamento dal disco")
    with open(PERCORSO_CORPUS_BASE, 'rb') as f:
        corpus_base = pickle.load(f)
    print(f"Caricato: {len(corpus_base):,} documenti")
else:
    print("Estrazione pagine necessarie dal corpus")
    corpus_base = {}
    for doc in tqdm(corpus_wikipedia, desc="Scansione corpus"):
        if doc['id'] in pagine_necessarie:
            corpus_base[doc['id']] = {
                'lines': doc['lines'],
                'text': doc['text']
            }
            if len(corpus_base) == len(pagine_necessarie):
                break

    print(f"Trovate: {len(corpus_base):,} / {len(pagine_necessarie):,}")
    print(f"Mancanti: {len(pagine_necessarie) - len(corpus_base):,}")

    with open(PERCORSO_CORPUS_BASE, 'wb') as f:
        pickle.dump(corpus_base, f)
    print(f"Salvato in '{PERCORSO_CORPUS_BASE}'")

Corpus base già salvato: caricamento dal disco
Caricato: 14,364 documenti


Conteggio di quante volte ogni pagina viene linkata dalle altre (informazione presente nel campo lines del dataset che è in formato *id\ttesto\tlink1\tlink2*

In [45]:
conteggio_link_entrata = Counter()

for titolo, doc in tqdm(corpus_base.items(), desc="Analisi link"):
    for riga in doc['lines'].split('\n'):
        parti = riga.split('\t')
        if len(parti) > 2:
            for link in parti[2:]:
                link = link.strip()
                if link and link in corpus_base:
                    conteggio_link_entrata[link] += 1

print(f"Pagine con almeno 1 link in entrata: {len(conteggio_link_entrata):,}")
print(f"Pagine senza link in entrata: {len(corpus_base) - len(conteggio_link_entrata):,}")
print()
print("10 pagine più linkate:")
for pagina, n in conteggio_link_entrata.most_common(10):
    print(f"  {pagina:<45} {n:>5} link")

Analisi link: 100%|██████████| 14364/14364 [00:00<00:00, 29518.86it/s]

Pagine con almeno 1 link in entrata: 1,420
Pagine senza link in entrata: 12,944

10 pagine più linkate:
  India                                           872 link
  California                                      827 link
  Canada                                          686 link
  London                                          672 link
  France                                          654 link
  NBC                                             638 link
  HBO                                             566 link
  Europe                                          543 link
  WWE                                             525 link
  England                                         507 link


Generazione di timestamp verosimile su ogni pagina. Viene fatto considerando che:
* pagine molto linkate --> pagine più vecchie;
* pagime poco linkate --> pagine più recenti

Il range si considera dal 2004 anno di nascita di Wikipedia al 2017 anno di creazione del dataset FEVER

In [31]:
random.seed(42)

DATA_INIZIO_WIKIPEDIA = datetime(2004, 1, 1)
DATA_CORPUS_FEVER = datetime(2017, 6, 1)
GIORNI_TOTALI = (DATA_CORPUS_FEVER - DATA_INIZIO_WIKIPEDIA).days
MAX_LINK = max(conteggio_link_entrata.values()) if conteggio_link_entrata else 1

In [36]:
def genera_timestamp(numero_link: int) -> str:
    ratio = numero_link / MAX_LINK  # 0.0 = poco linkato, 1.0 = molto linkato
    centro = int(GIORNI_TOTALI * (1 - ratio * 0.6)) # molto linkato → più vicino a 0 (2004)
    rumore = random.randint(-180, 180)   # ±6 mesi di variazione casuale
    giorni = max(0, min(GIORNI_TOTALI, centro + rumore))

    ore = random.randint(0, 23)
    minuti = random.randint(0, 59)
    secondi = random.randint(0, 59)

    data = DATA_INIZIO_WIKIPEDIA + timedelta(days=giorni, hours=ore, minutes=minuti, seconds=secondi)
    return data.strftime('%Y-%m-%dT%H:%M:%SZ')

In [39]:
print("Test generazione timestamp")
pagine_test = conteggio_link_entrata.most_common(3) + [(p, 0) for p in list(corpus_base.keys())[:3]]
for pagina, n_link in pagine_test:
    ts = genera_timestamp(n_link)
    print(f"  {pagina:<40} link={n_link:>5}  →  {ts}")

Test generazione timestamp
  India                                    link=  872  →  2009-10-06T11:10:23Z
  California                               link=  827  →  2009-10-13T06:42:17Z
  Canada                                   link=  686  →  2011-07-29T21:41:04Z
                                           link=    0  →  2017-06-01T20:10:34Z
  19_Kids_and_Counting                     link=    0  →  2017-04-07T05:29:24Z
  1985–86_NBA_season                       link=    0  →  2017-04-20T20:44:35Z


Salvataggio e assemblaggio del corpus

In [40]:
corpus_arricchito = {}

for titolo, doc in corpus_base.items():
    numero_link = conteggio_link_entrata.get(titolo, 0)
    corpus_arricchito[titolo] = {
        'lines': doc['lines'],
        'text': doc['text'],
        'numero_link_entrata': numero_link,
        'timestamp': genera_timestamp(numero_link)
    }

with open(PERCORSO_CORPUS_ARRICCHITO, 'wb') as f:
    pickle.dump(corpus_arricchito, f)

dimensione_mb = os.path.getsize(PERCORSO_CORPUS_ARRICCHITO) / 1024**2

Verifica finale

In [42]:
print("Verifica su 10 esempi")
for titolo, doc in list(corpus_arricchito.items())[:10]:
    prima_frase = estrai_frase(doc['lines'], 0)
    print(f"\nPagina:              {titolo}")
    print(f"  Link in entrata:   {doc['numero_link_entrata']}")
    print(f"  Timestamp:         {doc['timestamp']}")
    print(f"  Prima frase:       {prima_frase[:100]}")

Verifica su 10 esempi

Pagina:              
  Link in entrata:   0
  Timestamp:         2017-03-25T21:20:53Z
  Prima frase:       

Pagina:              19_Kids_and_Counting
  Link in entrata:   0
  Timestamp:         2016-12-31T07:52:02Z
  Prima frase:       19 Kids and Counting -LRB- formerly 17 Kids and Counting and 18 Kids and Counting -RRB- was an Ameri

Pagina:              1985–86_NBA_season
  Link in entrata:   0
  Timestamp:         2017-05-13T12:17:04Z
  Prima frase:       The 1985 -- 86 NBA season was the 40th season of the National Basketball Association .	National Bask

Pagina:              16th_IIFA_Awards
  Link in entrata:   0
  Timestamp:         2017-03-21T18:56:45Z
  Prima frase:       The 2015 IIFA Awards , officially known as the 16th International Indian Film Academy Awards ceremon

Pagina:              12_Play
  Link in entrata:   0
  Timestamp:         2017-05-13T06:41:31Z
  Prima frase:       12 Play is the debut studio album by American R&B and soul musician 